# 09. Bounded Foundry orchestration workbench

This notebook completes the local design of the optional outer-loop reasoning role. It consumes one response accepted by the evidence-analyst contract and produces one bounded proposal record for human review. It does not run Kaggriculture, score a game, edit policy source, invoke a model, deploy an agent, add tools, or create a workflow resource.

**Autonomy envelope:** all work in this notebook is local, deterministic, simulator-free, and cloud-dormant. The local evaluator remains authoritative for outcomes, audits, target binding, evidence writing, and experiment execution.

- [x] Define the accepted analyst-to-reasoning handoff.
- [x] Define and validate the full one-parameter proposal contract.
- [x] Exercise meaningful refusal cases.
- [x] Record the cloud-deployment boundary and pending decision.
- [ ] Human review: decide whether a second Foundry role is worth deploying.


## 1. Accepted analyst response becomes a narrow handoff

The predecessor is the promoted deterministic response gate, not free-form chat history. This section builds a redacted fixture, validates the response through the real local analyst contract, and exposes only packet binding, assessment, citations, limitations, and review status to the next role. Passing data forward does not grant execution authority.

In [1]:
from dataclasses import dataclass
import hashlib
import json
from pathlib import Path
import sys

project_root = Path.cwd()
package_root = project_root / "carrot-tuning-system"
if package_root.exists():
    sys.path.insert(0, str(package_root))

from analyst.contracts import EvidenceAnalystResponse, EvidenceCitation, validate_analyst_response

def canonical_sha256(value: object) -> str:
    """Bind the handoff to canonical redacted fixture bytes."""
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

REDACTED_PACKET = {"experiment_id": "candidate-evidence-orchestration-fixture-v1", "parameter_name": "seed_opportunity_value", "baseline_value": 20, "candidate_values": [10, 40, 60, 80], "interpretation_status": "one-seed smoke fixture; descriptive only", "all_audits_passed": True, "source_policy_mutation_authorized": False}
PACKET_SHA256 = canonical_sha256(REDACTED_PACKET)
ALLOWED_CITATIONS = (EvidenceCitation("packet.candidate_values", "declared values are 10, 40, 60, and 80"), EvidenceCitation("packet.interpretation_status", "one-seed smoke fixture; descriptive only"))
REQUIRED_LIMITATIONS = ("one-seed smoke fixture; descriptive only", "no source-policy mutation is authorized", "this handoff does not rank candidates or propose execution authority")
ANALYST_RESPONSE_JSON = {"analyst_contract_version": 1, "packet_sha256": PACKET_SHA256, "assessment": "The supplied packet is eligible for descriptive reasoning only; it does not establish a recommendation.", "citations": [{"packet_path": item.packet_path, "claim": item.claim} for item in ALLOWED_CITATIONS], "limitations": list(REQUIRED_LIMITATIONS), "review_status": "pending_human_review"}
VALIDATED_ANALYST_RESPONSE: EvidenceAnalystResponse = validate_analyst_response(ANALYST_RESPONSE_JSON, packet_sha256=PACKET_SHA256, allowed_citations=ALLOWED_CITATIONS, required_limitations=REQUIRED_LIMITATIONS)

@dataclass(frozen=True)
class AnalystHandoff:
    """The only accepted analyst facts permitted into the next role."""
    packet_sha256: str
    assessment: str
    citations: tuple[EvidenceCitation, ...]
    limitations: tuple[str, ...]
    review_status: str

HANDOFF = AnalystHandoff(VALIDATED_ANALYST_RESPONSE.packet_sha256, VALIDATED_ANALYST_RESPONSE.assessment, VALIDATED_ANALYST_RESPONSE.citations, VALIDATED_ANALYST_RESPONSE.limitations, VALIDATED_ANALYST_RESPONSE.review_status)


In [2]:
from IPython.display import Markdown, display
display(Markdown(f"### ✅ Analyst-to-reasoning handoff\n\n| Fact | Value |\n|:---|:---|\n| Packet binding | `{HANDOFF.packet_sha256[:12]}…` |\n| Predecessor gate | **passed** |\n| Input role | evidence analyst |\n| Input status | **{HANDOFF.review_status}** |\n| Citations / limitations | **{len(HANDOFF.citations)} / {len(HANDOFF.limitations)}** |\n| Simulator / source authority | **none** |\n\n> The handoff is a bounded data contract, not approval to run a proposal."))


### ✅ Analyst-to-reasoning handoff

| Fact | Value |
|:---|:---|
| Packet binding | `b23cfdb5f1af…` |
| Predecessor gate | **passed** |
| Input role | evidence analyst |
| Input status | **pending_human_review** |
| Citations / limitations | **2 / 3** |
| Simulator / source authority | **none** |

> The handoff is a bounded data contract, not approval to run a proposal.

## 2. Validate one bounded reasoning proposal

The original refinement architecture requires a structured proposal: an assessment, cited evidence, alternative explanations, one named parameter, one proposed value, a measurable prediction, a next measurement, limitations, and `pending_human_review`. The proposal envelope below is fixture input for contract testing; it is not a new approved experiment specification.

In [3]:
PROPOSAL_FIELDS = frozenset({"assessment", "evidence_cited", "alternative_explanations", "single_hypothesis", "next_measurement", "confidence_and_limits", "limitations", "review_status"})
HYPOTHESIS_FIELDS = frozenset({"parameter", "current_value", "proposed_value", "rationale", "prediction"})
ASSESSMENTS = frozenset({"inconclusive", "retain_baseline", "test_candidate", "investigate"})
PROPOSAL_INPUT_ENVELOPE = {"parameter": "seed_opportunity_value", "allowed_values": (10.0, 40.0, 60.0, 80.0), "authority": "fixture-only; a new experiment contract still requires human approval"}

def validate_reasoning_proposal(proposal: dict[str, object]) -> dict[str, object]:
    """Accept only one cited, bounded, pending-review proposal."""
    if set(proposal) != PROPOSAL_FIELDS:
        raise ValueError("proposal fields differ from the reasoning contract")
    if proposal["assessment"] not in ASSESSMENTS:
        raise ValueError("proposal assessment is not supported")
    if proposal["review_status"] != "pending_human_review":
        raise ValueError("proposal must remain pending human review")
    if not isinstance(proposal["evidence_cited"], list) or not proposal["evidence_cited"]:
        raise ValueError("proposal needs packet-backed evidence citations")
    allowed_pairs = {(item.packet_path, item.claim) for item in HANDOFF.citations}
    for citation in proposal["evidence_cited"]:
        if not isinstance(citation, dict) or set(citation) != {"packet_path", "claim"}:
            raise ValueError("proposal citation has an unsupported shape")
        if (citation["packet_path"], citation["claim"]) not in allowed_pairs:
            raise ValueError("proposal cites a claim outside the analyst handoff")
    if not isinstance(proposal["alternative_explanations"], list) or not proposal["alternative_explanations"]:
        raise ValueError("proposal must name at least one alternative explanation")
    hypothesis = proposal["single_hypothesis"]
    if not isinstance(hypothesis, dict) or set(hypothesis) != HYPOTHESIS_FIELDS:
        raise ValueError("proposal must contain exactly one complete hypothesis")
    if hypothesis["parameter"] != PROPOSAL_INPUT_ENVELOPE["parameter"]:
        raise ValueError("proposal parameter is outside the declared handoff envelope")
    if hypothesis["proposed_value"] not in PROPOSAL_INPUT_ENVELOPE["allowed_values"]:
        raise ValueError("proposal value is not in the declared fixture envelope")
    if not proposal["next_measurement"] or not proposal["confidence_and_limits"]:
        raise ValueError("proposal needs a next measurement and confidence limits")
    if not isinstance(proposal["limitations"], list) or not set(HANDOFF.limitations).issubset(proposal["limitations"]):
        raise ValueError("proposal dropped one or more analyst limitations")
    return dict(proposal)

PROPOSAL = validate_reasoning_proposal({"assessment": "test_candidate", "evidence_cited": [{"packet_path": item.packet_path, "claim": item.claim} for item in HANDOFF.citations], "alternative_explanations": ["The one-seed smoke fixture may reflect a realized threshold rather than a general effect.", "The observed delta may depend on opponent and seat coverage not present in this fixture."], "single_hypothesis": {"parameter": "seed_opportunity_value", "current_value": 20.0, "proposed_value": 40.0, "rationale": "The declared value is a bounded follow-up question, not a conclusion from this smoke fixture.", "prediction": "A held-out suite will determine whether the candidate changes audited paired outcomes."}, "next_measurement": "Run a separately approved held-out seed and seat suite with the same clean-audit requirement.", "confidence_and_limits": "This proposal is provisional and cannot establish a game outcome or policy recommendation.", "limitations": list(HANDOFF.limitations), "review_status": "pending_human_review"})


In [4]:
display(Markdown(f"### ✅ Bounded reasoning proposal\n\n| Fact | Value |\n|:---|:---|\n| Assessment | **{PROPOSAL['assessment']}** |\n| Parameter | `{PROPOSAL['single_hypothesis']['parameter']}` |\n| Proposed value | **{PROPOSAL['single_hypothesis']['proposed_value']}** |\n| Evidence citations | **{len(PROPOSAL['evidence_cited'])} accepted** |\n| Alternative explanations | **{len(PROPOSAL['alternative_explanations'])}** |\n| Simulator / source authority | **none** |\n| Review status | **{PROPOSAL['review_status']}** |\n\n> This is a proposal record. It is not a candidate result, experiment approval, or policy change."))


### ✅ Bounded reasoning proposal

| Fact | Value |
|:---|:---|
| Assessment | **test_candidate** |
| Parameter | `seed_opportunity_value` |
| Proposed value | **40.0** |
| Evidence citations | **2 accepted** |
| Alternative explanations | **2** |
| Simulator / source authority | **none** |
| Review status | **pending_human_review** |

> This is a proposal record. It is not a candidate result, experiment approval, or policy change.

## 3. Exercise the deterministic orchestration guards

The reasoning model is not the authority for structure or scope. These cases represent the failure modes that must be rejected before a future agent handoff: unknown fields, two-parameter reasoning, undeclared values, dropped limitations, unsupported citations, and claimed approval.

In [5]:
def expect_rejection(label: str, candidate: dict[str, object]) -> dict[str, str]:
    try:
        validate_reasoning_proposal(candidate)
    except ValueError as exc:
        return {"case": label, "status": "rejected_before_invocation", "reason": str(exc)}
    raise AssertionError(f"guard accepted unsafe case: {label}")

guard_cases = []
unknown_field = dict(PROPOSAL)
unknown_field["tool_request"] = "run simulator"
guard_cases.append(expect_rejection("unknown tool field", unknown_field))
two_parameters = dict(PROPOSAL)
two_parameters["single_hypothesis"] = dict(PROPOSAL["single_hypothesis"], parameter="pass_utility")
guard_cases.append(expect_rejection("second parameter", two_parameters))
undeclared_value = dict(PROPOSAL)
undeclared_value["single_hypothesis"] = dict(PROPOSAL["single_hypothesis"], proposed_value=999.0)
guard_cases.append(expect_rejection("undeclared candidate value", undeclared_value))
missing_limit = dict(PROPOSAL)
missing_limit["limitations"] = list(HANDOFF.limitations[:-1])
guard_cases.append(expect_rejection("dropped limitation", missing_limit))
claimed_approval = dict(PROPOSAL)
claimed_approval["review_status"] = "approved"
guard_cases.append(expect_rejection("claimed approval", claimed_approval))
GUARD_RESULTS = tuple(guard_cases)


In [6]:
guard_table = '\n'.join('| {} | **{}** |'.format(item['case'], item['status']) for item in GUARD_RESULTS)
display(Markdown('### ✅ Orchestration guard suite\n\n| Guard case | Result |\n|:---|:---|\n{0}\n\n| Guard cases exercised | **{1} / {1}** |\n\n> No unsafe proposal reached a model invocation or external system.'.format(guard_table, len(GUARD_RESULTS))))


### ✅ Orchestration guard suite

| Guard case | Result |
|:---|:---|
| unknown tool field | **rejected_before_invocation** |
| second parameter | **rejected_before_invocation** |
| undeclared candidate value | **rejected_before_invocation** |
| dropped limitation | **rejected_before_invocation** |
| claimed approval | **rejected_before_invocation** |

| Guard cases exercised | **5 / 5** |

> No unsafe proposal reached a model invocation or external system.

## 4. Cloud boundary and human decision record

The local workbench is complete enough to support a deployment decision, but it does not make that decision. The record below captures the conservative automatic choice to defer a second-agent deployment while preserving the alternatives, evidence, authority, and review task. A future cloud action must be represented by new reviewed notebook cells for desired state, discovery, request, apply, and readback.

- [ ] Review the proposal contract and decide whether to deploy an optional second no-tools Foundry role.
- [ ] If deployment is approved, review the exact target, model, cost, retention, and readback plan in a later cloud section.


In [7]:
ORCHESTRATION_DECISION = {
    "selected_option": "defer_second_agent_deployment",
    "alternatives_considered": ["keep the proposal-drafter local-only for the submission", "deploy one separate no-tools prompt agent after review", "build a larger multi-agent workflow with explicit orchestration"],
    "inputs": ["validated analyst response", "bounded proposal contract", "guard-suite results"],
    "authority": "local prototype envelope; no cloud authority granted",
    "side_effects": [],
    "cloud_action": "not_performed",
    "review_status": "pending_human_review",
}


In [8]:
display(Markdown(f"### ✅ Notebook 09 completion record\n\n| Fact | Value |\n|:---|:---|\n| Analyst handoff | **validated** |\n| Proposal contract | **validated** |\n| Guard cases | **{len(GUARD_RESULTS)} passed** |\n| Selected local option | `{ORCHESTRATION_DECISION['selected_option']}` |\n| Cloud action | **{ORCHESTRATION_DECISION['cloud_action']}** |\n| Review status | **{ORCHESTRATION_DECISION['review_status']}** |\n\n- [ ] Decide whether the optional proposal-drafter role should become a separately deployed Foundry agent.\n\n> Established: a reasoning role can propose one cited, bounded next question.\n> Not established: model quality, game outcome, experiment approval, or multi-agent deployment."))


### ✅ Notebook 09 completion record

| Fact | Value |
|:---|:---|
| Analyst handoff | **validated** |
| Proposal contract | **validated** |
| Guard cases | **5 passed** |
| Selected local option | `defer_second_agent_deployment` |
| Cloud action | **not_performed** |
| Review status | **pending_human_review** |

- [ ] Decide whether the optional proposal-drafter role should become a separately deployed Foundry agent.

> Established: a reasoning role can propose one cited, bounded next question.
> Not established: model quality, game outcome, experiment approval, or multi-agent deployment.

## Completion note

Notebook 09 now completes the local reasoning-contract prototype. It deliberately stops before deployment. The next step is a human choice: keep this as a documented architecture/demo artifact, or authorize a separate cloud component build for one no-tools proposal-drafter agent.